# Phase decomposition of generate-proof on Colab (native mode)
Run the cells **from the top, one at a time** (Shift+Enter). Runtime > Change runtime type must be **CPU**. Leave this tab open while the session runs (cell 5).
Choose the back end in cell 1: `BACKEND = ''` is the ark back end (the default of ZoKrates); `BACKEND = 'bellman'` is the second run.

In [ ]:
# 1. Upload TWO files when the button appears: zkp_v4d.zip and zkp_phases_kit_v2.zip
BACKEND = ''          # '' = ark (default).  For the second run set  BACKEND = 'bellman'
import os, zipfile, shutil, glob
from google.colab import files
up = files.upload()
for f in up:
    if f.endswith('.zip'): zipfile.ZipFile(f).extractall('/content')
for f in glob.glob('/content/zkp_phases/*'):
    if os.path.isfile(f): shutil.copy(f, '/content/zkp_v4/')
os.chdir('/content/zkp_v4')
os.environ['BACKEND'] = BACKEND
!chmod +x zokrates_timers zok_wrap.sh; ls -la zokrates_timers run_phases_linux.sh phases_extract.py zok_wrap.sh

In [ ]:
# 2. Connect Google Drive: results are copied there after every pass; an interrupted session is continued from that copy
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/zkp_phases_colab'
os.makedirs(BACKUP, exist_ok=True)
!cp -rn "$BACKUP"/phases_* /content/zkp_v4/ 2>/dev/null; ls -d phases_* 2>/dev/null

In [ ]:
# 3. Three-minute check: it must print READY, then a table of four circuits, and no line may start with !!!
!bash check_setup_linux.sh | tail -5 && bash run_phases_linux.sh quick 2>&1 | tail -12

In [ ]:
# 4. Start the full run in the background (it continues an unfinished run of the same back end by itself)
pre = 'phases_' + (BACKEND + '_' if BACKEND else '')
todo = [d for d in sorted(glob.glob(pre + '2*')) if not d.endswith('_quick') and (BACKEND or not d.startswith('phases_bellman'))
        and 'finished' not in open(d + '/raw_run.log', errors='replace').read()]
os.environ['RESUME'] = todo[-1] if todo else ''
os.environ['AFTER_PASS_HOOK'] = f'cp -r "$RES" "{BACKUP}/"; true'
print('continuing ' + todo[-1] if todo else 'starting a new run')
!if [ -z "$RESUME" ]; then unset RESUME; fi; nohup bash run_phases_linux.sh > run_phases.out 2>&1 &

In [ ]:
# 5. Watch it (prints every 5 minutes; keep this tab open until it says finished)
import subprocess, time
from IPython.display import clear_output
def running(): return subprocess.run(['pgrep', '-f', 'run_phases_linux.sh'], capture_output=True).returncode == 0
time.sleep(20)
while running():
    out = subprocess.run("grep -a '^\\[' run_phases.out | tail -3; tail -2 run_phases.out", shell=True, capture_output=True, text=True).stdout
    clear_output(wait=True); print(time.strftime('%H:%M:%S'), '\n' + out)
    time.sleep(300)
print('finished or stopped:'); print(subprocess.run('tail -30 run_phases.out', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 6. Pack the result folder and download it (a copy also goes to Drive)
d = sorted(d for d in glob.glob('phases_*') if os.path.isdir(d) and not d.endswith('_quick') and (('bellman' in d) == bool(BACKEND)))[-1]
shutil.make_archive(d + '_colab', 'zip', '.', d)
!cp "{d}_colab.zip" "$BACKUP"/
files.download(d + '_colab.zip')